# BQ6 — Customer Voice: Keyword Frequency in Negative Reviews

Input:  `02_data/processed/reviews_cleaned.csv`
Output: `keyword_frequency.csv` (import riêng vào Power BI cho trang
Customer Voice — bảng này KHÔNG có quan hệ với star schema, xem ghi chú
ở `dashboard_design.md`)

### Quy tắc nghiệp vụ (đã thống nhất ở BQ6.1)
- "Tiêu cực" = `rating <= 2`
- Chỉ xét review có nội dung (`has_content = True`)

### Cách tiếp cận
Thay vì chỉ đếm tần suất thô (dễ bị các từ chung chung như "sản phẩm",
"giao hàng" lấn át vì chúng xuất hiện nhiều ở MỌI loại review, không
riêng tiêu cực), notebook này tính thêm **độ đặc trưng
(distinctiveness)** — tỷ lệ tần suất trong nhóm tiêu cực so với tần suất
trong toàn bộ review có nội dung. Từ/cụm từ có độ đặc trưng cao là từ
thực sự "chỉ điểm" cho review tiêu cực, không phải từ xuất hiện đều ở
mọi review.

### Giới hạn (ghi nhận như đã làm với `subcategory_proxy`)
- Tokenize tiếng Việt dùng `underthesea` (nếu có cài đặt), fallback về
  tách theo khoảng trắng nếu chưa cài — kết quả kém chính xác hơn với
  từ ghép nếu dùng fallback.
- Đây là heuristic thống kê đơn giản, không phải sentiment model — cần
  đọc lại thủ công top kết quả trước khi đưa vào dashboard chính thức.


### 1. Setup & Load

In [ ]:
import re
from collections import Counter

import pandas as pd

try:
    from underthesea import word_tokenize
    TOKENIZER_AVAILABLE = True
except ImportError:
    TOKENIZER_AVAILABLE = False
    print("underthesea chưa được cài — dùng fallback tách theo khoảng trắng.")
    print("Cài đặt để có kết quả chính xác hơn: pip install underthesea --break-system-packages")

reviews = pd.read_csv("../02_data/processed/reviews_cleaned.csv")
print(reviews.shape)
reviews[["rating", "has_content", "content"]].head(3)


### 2. Lọc theo quy tắc nghiệp vụ BQ6.1

Tạo 2 tập:
- `negative_reviews`: nhóm chính cần phân tích (rating ≤ 2, có nội dung)
- `baseline_reviews`: toàn bộ review có nội dung (dùng làm mẫu số tính
  độ đặc trưng — không giới hạn rating)

In [ ]:
has_text = reviews["has_content"] == True

negative_reviews = reviews[has_text & (reviews["rating"] <= 2)].copy()
baseline_reviews = reviews[has_text].copy()

print(f"Negative reviews (rating<=2, có nội dung): {len(negative_reviews)}")
print(f"Baseline (toàn bộ review có nội dung):     {len(baseline_reviews)}")


### 3. Làm sạch văn bản

In [ ]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
PUNCT_NUMBER_PATTERN = re.compile(r"[0-9]+|[^\w\sÀ-ỹ]+")

def clean_text(text):
    text = str(text).lower()
    text = URL_PATTERN.sub(" ", text)
    text = PUNCT_NUMBER_PATTERN.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

negative_reviews["content_clean"] = negative_reviews["content"].apply(clean_text)
baseline_reviews["content_clean"] = baseline_reviews["content"].apply(clean_text)

negative_reviews[["content", "content_clean"]].head(3)


### 4. Tokenize tiếng Việt

In [ ]:
def tokenize(text):
    if TOKENIZER_AVAILABLE:
        # underthesea trả về từ ghép dạng "giao_hàng" -> đổi lại thành "giao hàng"
        return [tok.replace("_", " ") for tok in word_tokenize(text)]
    return text.split()

negative_reviews["tokens"] = negative_reviews["content_clean"].apply(tokenize)
baseline_reviews["tokens"] = baseline_reviews["content_clean"].apply(tokenize)

negative_reviews["tokens"].head(3)


### 5. Loại bỏ stopword

Kết hợp 2 nhóm:
- **Stopword tiếng Việt chung** (từ nối, đại từ, hư từ...)
- **Stopword riêng theo domain** (từ xuất hiện quá phổ biến trong bối
  cảnh review Tiki nói chung, không mang tính phân biệt: "sản phẩm",
  "shop", "tiki", "mình", "ạ"...) — danh sách này nên được bổ sung thêm
  sau khi xem thử kết quả ở Bước 7, nếu vẫn còn từ vô nghĩa lọt vào top.

In [ ]:
GENERIC_STOPWORDS = {
    "và", "là", "có", "được", "này", "đó", "rất", "cũng", "thì", "mà",
    "nhưng", "tôi", "mình", "của", "cho", "khi", "nên", "vì", "nếu",
    "các", "những", "một", "hai", "ba", "bị", "đã", "đang", "sẽ", "còn",
    "ở", "về", "để", "với", "từ", "trên", "dưới", "ra", "vào", "lên",
    "xuống", "không", "rồi", "nữa", "hơn", "quá", "lắm", "à", "ạ", "nhé",
    "nha", "ơi", "ừ", "vậy", "thế", "nào", "gì", "sao", "ai", "đây",
}

DOMAIN_STOPWORDS = {
    "sản phẩm", "shop", "tiki", "hàng", "mua", "dùng", "sử dụng",
    "giao hàng", "đóng gói",  # loại riêng nếu muốn giữ lại thì bỏ khỏi set này
}

# Lưu ý: "giao hàng" và "đóng gói" đang bị loại vì quá chung chung, NHƯNG
# nếu review thực tế phàn nàn "giao hàng CHẬM" thì cụm bigram "hàng chậm"
# vẫn được giữ (vì bigram khác token đơn) -> không mất thông tin quan trọng.

STOPWORDS = GENERIC_STOPWORDS | DOMAIN_STOPWORDS

def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

negative_reviews["tokens_clean"] = negative_reviews["tokens"].apply(remove_stopwords)
baseline_reviews["tokens_clean"] = baseline_reviews["tokens"].apply(remove_stopwords)


### 6. Đếm tần suất unigram + bigram

Bigram (cụm 2 từ liên tiếp) thường dễ đọc và có ý nghĩa hơn unigram đơn
lẻ cho việc trình bày dashboard (ví dụ "hàng lỗi" rõ nghĩa hơn "lỗi"
đứng một mình).

In [ ]:
def get_ngrams(tokens, n):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def count_ngrams(df, n):
    counter = Counter()
    for tokens in df["tokens_clean"]:
        counter.update(get_ngrams(tokens, n))
    return counter

neg_unigrams = count_ngrams(negative_reviews, 1)
neg_bigrams = count_ngrams(negative_reviews, 2)
base_unigrams = count_ngrams(baseline_reviews, 1)
base_bigrams = count_ngrams(baseline_reviews, 2)

print(f"Unique unigrams (negative): {len(neg_unigrams)}")
print(f"Unique bigrams (negative):  {len(neg_bigrams)}")


### 7. Tính độ đặc trưng (distinctiveness)

```
distinctiveness = (freq_negative / tổng số từ trong negative)
                 / (freq_baseline / tổng số từ trong baseline)
```

Tỷ lệ > 1 nghĩa là từ/cụm từ đó xuất hiện trong review tiêu cực **nhiều
hơn tỷ lệ trung bình** so với toàn bộ review — càng cao càng "chỉ điểm"
đúng cho vấn đề tiêu cực. Thêm hệ số làm mượt (`smoothing`) để tránh
chia cho 0 khi 1 từ chưa từng xuất hiện ở baseline.

In [ ]:
def build_keyword_table(neg_counter, base_counter, ngram_type, min_freq=5, smoothing=1):
    total_neg = sum(neg_counter.values())
    total_base = sum(base_counter.values())

    rows = []
    for keyword, freq_neg in neg_counter.items():
        if freq_neg < min_freq:
            continue  # loại cụm từ quá hiếm, dễ gây nhiễu tỷ lệ
        freq_base = base_counter.get(keyword, 0)

        rate_neg = freq_neg / total_neg
        rate_base = (freq_base + smoothing) / (total_base + smoothing)

        rows.append({
            "keyword": keyword,
            "ngram_type": ngram_type,
            "frequency_negative": freq_neg,
            "frequency_baseline": freq_base,
            "distinctiveness_score": round(rate_neg / rate_base, 2),
        })
    return pd.DataFrame(rows)

df_unigram = build_keyword_table(neg_unigrams, base_unigrams, "unigram")
df_bigram = build_keyword_table(neg_bigrams, base_bigrams, "bigram")

keyword_df = pd.concat([df_unigram, df_bigram], ignore_index=True)
keyword_df = keyword_df.sort_values("distinctiveness_score", ascending=False)

print(f"Tổng số keyword (sau khi lọc min_freq): {len(keyword_df)}")
keyword_df.head(20)


### 8. Xem trước top kết quả — kiểm tra thủ công trước khi lưu

**Bước quan trọng, đừng bỏ qua**: đọc qua danh sách này, nếu thấy từ vô
nghĩa/lỗi tokenize lọt vào top, quay lại Bước 5 bổ sung stopword rồi
chạy lại từ đó.

In [ ]:
pd.set_option("display.max_rows", 30)
keyword_df[keyword_df["frequency_negative"] >= 10].head(30)


### 9. Lưu kết quả

In [ ]:
TOP_N = 30

output_df = (
    keyword_df[keyword_df["frequency_negative"] >= 10]
    .sort_values("distinctiveness_score", ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)

output_df.to_csv("keyword_frequency.csv", index=False, encoding="utf-8-sig")

print(f"Đã lưu {len(output_df)} keyword vào keyword_frequency.csv")
output_df


### 10. Ghi chú cho Power BI

Import `keyword_frequency.csv` như 1 bảng độc lập, **không** tạo quan hệ
với star schema (bảng này là kết quả tính sẵn ở cấp toàn dataset, không
lọc được theo slicer — đã ghi chú trong `dashboard_design.md`).

Bar chart gợi ý: Axis = `keyword`, Values = `frequency_negative` (dễ
hiểu với người đọc không quen thuật ngữ thống kê) hoặc
`distinctiveness_score` (chính xác hơn về mặt phân tích) — có thể thêm
2 bản bar chart cho cả 2 cách sort, hoặc dùng 1 slicer riêng để người
xem tự chọn tiêu chí.